In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, KNNWithMeans
from surprise.model_selection import train_test_split
from surprise import accuracy
import warnings
warnings.filterwarnings('ignore')

# Load the ratings data with pandas
ratings_df = pd.read_csv("../artifacts/rating.csv")

print("=" * 60)
print("DATA EXPLORATION")
print("=" * 60)
print(f"\nDataset shape: {ratings_df.shape}")
print(f"\nFirst few rows:")
print(ratings_df.head(10))
print(f"\nData types:")
print(ratings_df.dtypes)
print(f"\nBasic statistics:")
print(ratings_df.describe())
print(f"\nNumber of unique users: {ratings_df['userId'].nunique()}")
print(f"Number of unique movies: {ratings_df['movieId'].nunique()}")
print(f"Rating range: {ratings_df['rating'].min()} - {ratings_df['rating'].max()}")
print(f"Sparsity: {(1 - len(ratings_df) / (ratings_df['userId'].nunique() * ratings_df['movieId'].nunique())) * 100:.2f}%")

# Load training and test sets
print("\n" + "=" * 60)
print("LOADING TRAINING AND TEST SETS")
print("=" * 60)

train_df = pd.read_csv("../artifacts/ratings_train.csv")
test_df = pd.read_csv("../artifacts/ratings_test.csv")

print(f"\nTraining set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"Training set rating range: {train_df['rating'].min()} - {train_df['rating'].max()}")
print(f"Test set rating range: {test_df['rating'].min()} - {test_df['rating'].max()}")

DATA EXPLORATION

Dataset shape: (20000263, 4)

First few rows:
   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40
5       1      112     3.5  2004-09-10 03:09:00
6       1      151     4.0  2004-09-10 03:08:54
7       1      223     4.0  2005-04-02 23:46:13
8       1      253     4.0  2005-04-02 23:35:40
9       1      260     4.0  2005-04-02 23:33:46

Data types:
userId         int64
movieId        int64
rating       float64
timestamp     object
dtype: object

Basic statistics:
             userId       movieId        rating
count  2.000026e+07  2.000026e+07  2.000026e+07
mean   6.904587e+04  9.041567e+03  3.525529e+00
std    4.003863e+04  1.978948e+04  1.051989e+00
min    1.000000e+00  1.000000e+00  5.000000e-01
25%    3.439500e+04  9.020000e+02  3.0

In [2]:
# Prepare data for Surprise library
print("\n" + "=" * 60)
print("PREPARING DATA FOR SURPRISE LIBRARY")
print("=" * 60)

# Define the reader to specify the rating scale
reader = Reader(rating_scale=(ratings_df['rating'].min(), ratings_df['rating'].max()))

# Load training data
trainset = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader).build_full_trainset()

# Create testset from the test dataframe
testset = Dataset.load_from_df(test_df[['userId', 'movieId', 'rating']], reader).build_full_trainset().build_testset()

print(f"Training set (Surprise format) - n_users: {trainset.n_users}, n_items: {trainset.n_items}")
print(f"Test set (Surprise format) - n_ratings: {len(testset)}")

# Initialize and train the User-User Collaborative Filtering model
print("\n" + "=" * 60)
print("TRAINING USER-USER COLLABORATIVE FILTERING MODEL")
print("=" * 60)

# KNNWithMeans uses Pearson correlation by default for similarity computation
# sim_options allows us to configure the similarity metric
algo = KNNWithMeans(
    k=20,  # Number of nearest neighbors
    sim_options={
        'name': 'pearson',  # Use Pearson correlation
        'user_based': True  # User-based collaborative filtering
    },
    verbose=True
)

print("\nTraining the model...")
algo.fit(trainset)

# Make predictions on the test set
print("\n" + "=" * 60)
print("EVALUATING MODEL")
print("=" * 60)

predictions = algo.test(testset)

# Calculate RMSE and MAE
rmse = accuracy.rmse(predictions, verbose=True)
mae = accuracy.mae(predictions, verbose=True)

print(f"\nModel Performance:")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")

# Example: Make a recommendation for a specific user
print("\n" + "=" * 60)
print("EXAMPLE RECOMMENDATION")
print("=" * 60)

# Get a sample user from the test set
sample_user_id = test_df['userId'].iloc[0]
sample_user_movies = set(train_df[train_df['userId'] == sample_user_id]['movieId'].unique())

# Find movies the user hasn't rated yet
all_movies = set(train_df['movieId'].unique())
unrated_movies = list(all_movies - sample_user_movies)[:10]  # Top 10 unrated movies

print(f"\nMaking predictions for user {sample_user_id}:")
print(f"User has rated {len(sample_user_movies)} movies")
print(f"Predicting ratings for {len(unrated_movies)} unrated movies:\n")

recommendations = []
for movie_id in unrated_movies:
    pred = algo.predict(sample_user_id, movie_id)
    recommendations.append((pred.iid, pred.est))
    print(f"Movie {pred.iid}: Predicted rating = {pred.est:.2f}")


PREPARING DATA FOR SURPRISE LIBRARY


: 